# Notebook ქმნის getdata.raw.MotogpTest ცხრილს motogp_dataset_v2.csv ფაილის გამოყენებით

## ვქმნი getdata.raw.MotogpTest ცხრილს და ვსაზღვრავ ცხრილის ზუსტ სქემას

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS getdata.raw")

spark.sql("""
    CREATE TABLE IF NOT EXISTS getdata.raw.MotogpTest (
        SpeedKmH DECIMAL(10,2),
        CornerRadiusM DECIMAL(10,2),
        TrackGrip DECIMAL(10,4),
        BikeMassKg DECIMAL(10,2),
        TrackTemperature DECIMAL(10,2),
        TrackBankingDeg DECIMAL(10,2),
        TireType STRING,
        LeanAngleDeg DECIMAL(10,2)
    )
""")

## ნედლი CSV ფაილის ჩატვირთვა და ცხრილში ჩაწერა
### ვკითხულობთ CSV ფაილს და ვარეგულირებთ ტიპებს მონაცემთა ბაზის სქემის შესაბამისად.

In [0]:
import os

# ვიღებთ ფაილის ზუსტ გზას მიმდინარე სივრცეში
file_path = f"file://{os.path.abspath('RawCsvData/motogp_dataset_v2.csv')}"

df_raw = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(file_path)
)

# ვარეგულირებთ ათწილადების სიზუსტეს (Rounding) ცხრილში ჩაწერამდე
from pyspark.sql.functions import col, round

df_formatted = df_raw.select(
    round(col("SpeedKmH"), 2).cast("DECIMAL(10,2)").alias("SpeedKmH"),
    round(col("CornerRadiusM"), 2).cast("DECIMAL(10,2)").alias("CornerRadiusM"),
    round(col("TrackGrip"), 4).cast("DECIMAL(10,4)").alias("TrackGrip"),
    round(col("BikeMassKg"), 2).cast("DECIMAL(10,2)").alias("BikeMassKg"),
    round(col("TrackTemperature"), 2).cast("DECIMAL(10,2)").alias("TrackTemperature"),
    round(col("TrackBankingDeg"), 2).cast("DECIMAL(10,2)").alias("TrackBankingDeg"),
    col("TireType").cast("STRING").alias("TireType"),
    round(col("LeanAngleDeg"), 2).cast("DECIMAL(10,2)").alias("LeanAngleDeg")
)

# მონაცემების შენახვა ცხრილში
df_formatted.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("getdata.raw.MotogpTest")

### მონაცემების შემოწმება

In [0]:
display(spark.sql("SELECT * FROM getdata.raw.MotogpTest LIMIT 5"))